In [6]:
!pip install langchain langchain-community langchain-google-genai wikipedia

In [7]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableMap, RunnableLambda

In [8]:
import os
from google.colab import userdata
from google import genai
# Load API key from Colab Secrets into environment variable
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [9]:
# Set API key
# os.environ["GOOGLE_API_KEY"] = "your_google_api_key"

# Gemini LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Wikipedia tool
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

# Step 1 — Retrieve context
retriever = RunnableLambda(lambda topic: wiki_tool.run(topic))

# Step 2 — Summarization
summary_prompt = ChatPromptTemplate.from_template(
"""
Summarize the following information in 5 sentences.

Information:
{context}
"""
)

summary_chain = summary_prompt | llm

# Step 3 — Quiz generation
quiz_prompt = ChatPromptTemplate.from_template(
"""
Based on the summary below, generate 3 quiz questions.

Summary:
{summary}
"""
)

quiz_chain = quiz_prompt | llm


In [10]:

# Full pipeline
pipeline = (
    RunnableMap({"context": retriever})
    | summary_chain
    | RunnableLambda(lambda msg: {"summary": msg.content})
    | quiz_chain
)

# Run pipeline
topic = input("Enter a topic: ")

result = pipeline.invoke(topic)

print("\nQuiz Questions:\n")
print(result.content)

Enter a topic: Nikolai Tesla

Quiz Questions:

Here are 3 quiz questions based on the summary:

1.  What was Nikola Tesla primarily known for contributing to?
2.  What was the name of Nikola Tesla's unfinished project for worldwide wireless power distribution?
3.  What SI unit of magnetic flux density is named in Nikola Tesla's honor?
